<a href="https://colab.research.google.com/github/markajbell/BH/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Upload Data

In [1]:
# Upload data from github
!wget https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/data/data.zip

--2025-05-23 12:53:59--  https://raw.githubusercontent.com/markajbell/BH/refs/heads/main/data/data.zip
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 82583 (81K) [application/zip]
Saving to: ‘data.zip’

data.zip            100%[===================>]  80.65K  --.-KB/s    in 0.02s   

2025-05-23 12:53:59 (3.33 MB/s) - ‘data.zip’ saved [82583/82583]



In [2]:
#Create the data folder and unzip
!mkdir/content/data
!unzip data.zip -d /content/data

/bin/bash: line 1: mkdir/content/data: No such file or directory
Archive:  data.zip
  inflating: /content/data/bh-10.json  
  inflating: /content/data/bh-6-all_tier_0.json  
  inflating: /content/data/bh-7.json  
  inflating: /content/data/bh_all_tier_0-formatted.json  
  inflating: /content/data/bh-all_tier_0.json  
  inflating: /content/data/bh-graph.json  
  inflating: /content/data/bh-unconstrained.json  
  inflating: /content/data/tier0.json  


# Import JSON and conver to pd

In [24]:
# import JSON and load into a pandas dataframe (df_all)

!pip install pandas

import pandas as pd
import json

with open('data/bh-unconstrained.json', 'r') as f:
    data_all = json.load(f)

df_all = pd.DataFrame(data_all)

# Display the first few rows of the DataFrame to verify
print(df_all.head())


# Create Nodes and Edges from df_all

In [15]:
# from df_all['data'].keys() extract the 'nodes' into a seperate dataframe called nodes_all.

import pandas as pd
pd.set_option("display.max_colwidth", None)
#transpose the columns
nodes_all = pd.DataFrame(df_all['data']['nodes']).T
#label columns
nodes_all.columns = ['label', 'kind', 'OID', 'istier0', 'isOwned', 'lastSeen']
#convert the nodes index to int
nodes_all.index = nodes_all.index.astype(int)
#rename the index to node_id
nodes_all = nodes_all.reset_index().rename(columns={'index': 'node_id'})
#print(nodes_all)
nodes_all.head()

,node_id,label,kind,OID,istier0,isOwned,lastSeen
0,5,DOMAIN ADMINS@MYLAB.LOCAL,Group,S-1-5-21-1333368235-828418111-1849936505-512,True,False,2025-05-13T13:28:49.512Z
1,6,BDEWAPPS1000000.MYLAB.LOCAL,Computer,S-1-5-21-1333368235-828418111-1849936505-4092,False,False,2025-05-13T13:28:36.181237651Z
2,5075,OGCWLPT1000000.MYLAB.LOCAL,Computer,S-1-5-21-1333368235-828418111-1849936505-4093,False,False,2025-05-13T13:28:36.181237651Z
3,5076,BDEWLPT1000000.MYLAB.LOCAL,Computer,S-1-5-21-1333368235-828418111-1849936505-4094,False,False,2025-05-13T13:28:36.181237651Z
4,5077,HREWVIR1000000.MYLAB.LOCAL,Computer,S-1-5-21-1333368235-828418111-1849936505-4095,False,False,2025-05-13T13:28:36.181237651Z


In [16]:
# create a seperate dataframe from df_all using the edges key

import pandas as pd
edges_all = pd.DataFrame(df_all['data']['edges'])
print(edges_all)

     source target       label        kind                        lastSeen
0         5   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
1         6   6264    MemberOf    MemberOf  2025-05-13T13:28:36.181502638Z
2      6264   5186    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
3      5186   5483  GenericAll  GenericAll  2025-05-13T13:28:36.832245798Z
4      5483   5179    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
...     ...    ...         ...         ...                             ...
3953   5178   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
3954   6689   6259    MemberOf    MemberOf  2025-05-13T13:28:44.995877259Z
3955   6259   5168  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z
3956   5168   5178  GenericAll  GenericAll  2025-05-13T13:28:44.995877259Z
3957   5178   5173  GenericAll  GenericAll  2025-05-13T13:28:36.181502638Z

[3958 rows x 5 columns]


In [25]:
# Apply the weights based on the 'label' column
def assign_weight(label):
    if label == "GenericAll":
        return 9
    elif label == "Owns":
        return 10
    elif label == "GenericWrite":
        return 2
    elif label == "AllExtendedRights":
        return 6
    elif label == "CanRDP":
        return 2
    elif label == "Contains":
        return 2
    elif label == "DCSync":
        return 8
    elif label == "WriteDacl":
        return 5
    elif label == "WriteOwner":
        return 7
    elif label == "AddKeyCredentialLink":
        return 6
    elif label == "AdminTo":
        return 8
    elif label == "MemberOf":
        return 1
    elif label == "CanPSRemote":
        return 2
    elif label == "ExecuteDCOM":
        return 2
    elif label == "GPLink":
        return 3
    elif label == "tier0":
        return 0

    else:
        return 1 # Default weight for other labels

edges_all['weight'] = edges_all['label'].apply(assign_weight)

# Display the updated edges_all DataFrame with the 'weight' column
print("Edges_all DataFrame with weight column:")
edges_all

Edges_all DataFrame with weight column:


,source,target,label,kind,lastSeen,weight
0,5,5173,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
1,6,6264,MemberOf,MemberOf,2025-05-13T13:28:36.181502638Z,1
2,6264,5186,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
3,5186,5483,GenericAll,GenericAll,2025-05-13T13:28:36.832245798Z,9
4,5483,5179,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
...,...,...,...,...,...,...
3953,5178,5173,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
3954,6689,6259,MemberOf,MemberOf,2025-05-13T13:28:44.995877259Z,1
3955,6259,5168,GenericAll,GenericAll,2025-05-13T13:28:36.181502638Z,9
3956,5168,5178,GenericAll,GenericAll,2025-05-13T13:28:44.995877259Z,9


In [26]:
# convert edges_all['source') to int
edges_all['source'] = edges_all['source'].astype(int)
edges_all['target'] = edges_all['target'].astype(int)

# Verify the data types
print("\nData types after conversion:")
edges_all.dtypes


Data types after conversion:


,0
source,int64
target,int64
label,object
kind,object
lastSeen,object
weight,int64


# PageRank Analysis

In [37]:
# Install network x and import

!pip install networkx
import networkx as nx

# Create a directed graph
G = nx.DiGraph()

# Add nodes to the graph
# You can add nodes with attributes from your nodes_all DataFrame if needed
for index, row in nodes_all.iterrows():
    G.add_node(row['node_id'], label=row['label'], kind=row['kind'], OID=row['OID'], istier0=row['istier0'], isOwned=row['isOwned'], lastSeen=row['lastSeen'])

# Add edges to the graph with weights
for index, row in edges_all.iterrows():
    G.add_edge(row['source'], row['target'], weight=row['weight'])

# Calculate PageRank
# alpha is the damping parameter, commonly set to 0.85
pagerank_scores = nx.pagerank(G, alpha=0.5, weight='weight')

# You can now access the PageRank score for each node
# For example, to see the scores:
print("\nPageRank scores:")
for node, score in pagerank_scores.items():
    #print(f"Node {node}: {score:.4f}")
    pass
# To add the PageRank scores back to your nodes_all DataFrame:
nodes_all['pagerank'] = nodes_all['node_id'].map(pagerank_scores)

print("\nNodes_all DataFrame with PageRank scores:")
print(nodes_all[['node_id', 'label', 'pagerank']].head(20))

# sort the nodes by PageRank to find the most important nodes
#print("\nTop 10 nodes by PageRank:")
#print(nodes_all.sort_values(by='pagerank', ascending=False).head(12))


PageRank scores:

Nodes_all DataFrame with PageRank scores:
    node_id                        label  pagerank
0         5    DOMAIN ADMINS@MYLAB.LOCAL  0.002081
1         6  BDEWAPPS1000000.MYLAB.LOCAL  0.000475
2      5075   OGCWLPT1000000.MYLAB.LOCAL  0.000475
3      5076   BDEWLPT1000000.MYLAB.LOCAL  0.000475
4      5077   HREWVIR1000000.MYLAB.LOCAL  0.000475
5      5078  GOOWSECS1000000.MYLAB.LOCAL  0.000475
6      5079   GOOWVIR1000000.MYLAB.LOCAL  0.000475
7      5080   SECWLPT1000000.MYLAB.LOCAL  0.000475
8      5081  HREWAPPS1000000.MYLAB.LOCAL  0.000475
9      5082   SECWVIR1000000.MYLAB.LOCAL  0.000475
10     5083   BDEWWKS1000000.MYLAB.LOCAL  0.000475
11     5084  GOOWWEBS1000000.MYLAB.LOCAL  0.000475
12     5085   AZRWLPT1000000.MYLAB.LOCAL  0.000950
13     5086   FINWWKS1000000.MYLAB.LOCAL  0.000475
14     5087   OGCWWKS1000000.MYLAB.LOCAL  0.000950
15     5088   ITSWWKS1000000.MYLAB.LOCAL  0.000475
16     5089   FSRWLPT1000000.MYLAB.LOCAL  0.000475
17     5090  OGCWDBAS

In [40]:
# Sort the DataFrame by pagerank in descending order
sorted_pagerank = nodes_all.sort_values(by='pagerank', ascending=False)

# Print the sorted DataFrame
print("\nNodes sorted by PageRank (highest to lowest):")
print(sorted_pagerank[['node_id', 'label', 'pagerank', 'kind']])


Nodes sorted by PageRank (highest to lowest):
     node_id                               label  pagerank      kind
100     5173                  DCTEST.MYLAB.LOCAL  0.051142  Computer
105     5178  MA-VACACIONE-DISTLIST1@MYLAB.LOCAL  0.024331     Group
775     6264        DOMAIN COMPUTERS@MYLAB.LOCAL  0.022034     Group
104     5177        DE-DAS-DISTLIST1@MYLAB.LOCAL  0.020857     Group
106     5179        RO-ARJ-DISTLIST1@MYLAB.LOCAL  0.017567     Group
..       ...                                 ...       ...       ...
455     5848               SHAWN_FOX@MYLAB.LOCAL  0.000475      User
456     5849      CHRISTOPER_SIMPSON@MYLAB.LOCAL  0.000475      User
457     5850          NADINE_HOLLAND@MYLAB.LOCAL  0.000475      User
458     5853             GAIL_ALSTON@MYLAB.LOCAL  0.000475      User
443     5819          CANDACE_ASHLEY@MYLAB.LOCAL  0.000475      User

[1106 rows x 4 columns]
